# Prod MLflow Tracking V3 API Display

Pulls the public telemetry export API and displays the result directly in the notebook.

In [0]:
import datetime as dt
import json
import os
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd

try:
    from pyspark.sql import functions as F
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType
except ImportError:
    F = None
    BooleanType = DoubleType = LongType = StringType = StructField = StructType = None


class _LocalWidgets:
    def __init__(self):
        self._defaults = {}

    def text(self, name: str, default_value: str, label: str | None = None) -> None:
        self._defaults[name] = default_value

    def dropdown(self, name: str, default_value: str, choices: list[str], label: str | None = None) -> None:
        self._defaults[name] = default_value

    def get(self, name: str) -> str:
        env_name = f"NOTEBOOK_{name.upper()}"
        return os.environ.get(env_name, self._defaults.get(name, ""))


class _LocalDbutils:
    widgets = _LocalWidgets()


try:
    dbutils
except NameError:
    dbutils = _LocalDbutils()

try:
    spark
except NameError:
    spark = None


APP_ENV = "prod"
BASE_URL_DEFAULT = "https://apim-external-cub3dqcrgsdnebcb.a01.azurefd.net/benefits-prd/plan-list/"

dbutils.widgets.text("subscription_key", "651d20d2dc0243b19ccc0762e2246e67")
dbutils.widgets.text("base_url", BASE_URL_DEFAULT)
dbutils.widgets.text("start_date", dt.date.today().isoformat())
dbutils.widgets.text("end_date", dt.date.today().isoformat())
dbutils.widgets.text("limit", "10000")
dbutils.widgets.dropdown("include_text", "false", ["false", "true"])
dbutils.widgets.text("app_env_filter", "")

SUBSCRIPTION_KEY = dbutils.widgets.get("subscription_key").strip()
BASE_URL = dbutils.widgets.get("base_url").strip().rstrip("/") + "/"
START_DATE = dt.date.fromisoformat(dbutils.widgets.get("start_date").strip())
END_DATE = dt.date.fromisoformat(dbutils.widgets.get("end_date").strip())
LIMIT = max(1, min(int(dbutils.widgets.get("limit").strip() or "10000"), 10000))
INCLUDE_TEXT = dbutils.widgets.get("include_text").strip().lower() == "true"
APP_ENV_FILTER = dbutils.widgets.get("app_env_filter").strip()

if not SUBSCRIPTION_KEY:
    raise ValueError("Set the subscription_key widget before running the notebook.")
if END_DATE < START_DATE:
    raise ValueError("end_date must be on or after start_date.")


def _fetch_page(event_date: dt.date, cursor: str | None = None) -> dict:
    params = {
        "dataset": "mlflow_tracking_v3",
        "event_date": event_date.isoformat(),
        "limit": str(LIMIT),
        "include_text": "true" if INCLUDE_TEXT else "false",
    }
    if APP_ENV_FILTER:
        params["app_env"] = APP_ENV_FILTER
    if cursor:
        params["cursor"] = cursor

    url = BASE_URL + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"APIM request failed for {event_date.isoformat()} with {exc.code}: {body[:1000]}"
        ) from exc


def _validate_export_payload(payload: dict, event_date: dt.date) -> None:
    if not isinstance(payload, dict):
        raise RuntimeError(
            f"Unexpected response type for {event_date.isoformat()}: {type(payload).__name__}"
        )
    if "items" not in payload or "has_more" not in payload:
        preview = json.dumps(payload, ensure_ascii=False)[:1000]
        raise RuntimeError(
            "Unexpected export response shape for "
            f"{event_date.isoformat()}: keys={sorted(payload.keys())} preview={preview}"
        )
    if not isinstance(payload.get("items"), list):
        raise RuntimeError(
            f"Unexpected items type for {event_date.isoformat()}: {type(payload.get('items')).__name__}"
        )


def _infer_spark_type(values: list[object]):
    if StringType is None:
        return None
    for value in values:
        if value is None:
            continue
        if isinstance(value, bool):
            return BooleanType()
        if isinstance(value, int) and not isinstance(value, bool):
            return LongType()
        if isinstance(value, float):
            return DoubleType()
        return StringType()
    return StringType()


def _items_to_df(items: list[dict]):
    seen = set()
    columns = []
    for item in items:
        for key in item.keys():
            if key == "cached_input_tokens" or key in seen:
                continue
            seen.add(key)
            columns.append(key)
    if "error_message" in seen:
        columns = [column for column in columns if column != "error_message"] + ["error_message"]
    if "app_env" in seen:
        user_columns = [column for column in ("user_id", "user_name") if column in seen]
        if user_columns:
            columns = [column for column in columns if column not in {"user_id", "user_name"}]
            insert_at = columns.index("app_env") + 1
            columns[insert_at:insert_at] = user_columns
    if spark is None:
        return pd.DataFrame([{column: item.get(column) for column in columns} for item in items], columns=columns)
    schema = StructType(
        [StructField(column, _infer_spark_type([item.get(column) for item in items]), True) for column in columns]
    )
    rows = [
        {column: item.get(column) for column in columns}
        for item in items
    ]
    return spark.createDataFrame(rows, schema=schema)


def _drop_all_null_columns(df):
    if isinstance(df, pd.DataFrame):
        return df.dropna(axis=1, how="all")
    if not df.columns:
        return df
    non_null_flags = (
        df.agg(
            *[
                F.max(F.when(F.col(column).isNotNull(), F.lit(1)).otherwise(F.lit(0))).alias(column)
                for column in df.columns
            ]
        )
        .collect()[0]
        .asDict()
    )
    keep_columns = [column for column in df.columns if non_null_flags.get(column)]
    return df.select(*keep_columns) if keep_columns else df


In [0]:
total_rows = 0
total_pages = 0
day_summaries: list[dict] = []
api_df = None

day = START_DATE
while day <= END_DATE:
    day_rows = 0
    cursor = None
    while True:
        payload = _fetch_page(day, cursor=cursor)
        _validate_export_payload(payload, day)
        items = payload.get("items") or []
        total_pages += 1

        if items:
            batch_df = _items_to_df(items)
            if api_df is None:
                api_df = batch_df
            elif spark is None:
                api_df = pd.concat([api_df, batch_df], ignore_index=True, sort=False)
            else:
                api_df = api_df.unionByName(batch_df, allowMissingColumns=True)
            day_rows += len(items)
            total_rows += len(items)

        if not payload.get("has_more"):
            break
        cursor = payload.get("next_cursor")
        if not cursor:
            raise RuntimeError(f"Missing next_cursor for {day.isoformat()} despite has_more=true.")

    day_summaries.append({"event_date": day.isoformat(), "row_count": day_rows})
    print(f"{day.isoformat()}: {day_rows} rows")
    day += dt.timedelta(days=1)

summary = {
    "app_env": APP_ENV,
    "start_date": START_DATE.isoformat(),
    "end_date": END_DATE.isoformat(),
    "total_rows": total_rows,
    "total_pages": total_pages,
}
print(json.dumps(summary, indent=2))


2026-04-15: 14 rows
{
  "app_env": "prod",
  "start_date": "2026-04-15",
  "end_date": "2026-04-15",
  "total_rows": 14,
  "total_pages": 1
}


In [0]:
if spark is None:
    display(pd.DataFrame(day_summaries))
else:
    display(spark.createDataFrame(day_summaries))

if api_df is None:
    print("No rows returned from the API for the requested window.")
else:
    display(_drop_all_null_columns(api_df))

event_date,row_count
2026-04-15,14


tracking_id,event_time,question_id,session_id,response_id,facets_product_id,response_status,response_time_sec,retry_count,input_chars,output_chars,mlflow_run_id,mlflow_experiment,mlflow_tracking_uri,model_name,app_env,user_id,user_name,request_text_preview,request_text_sha256,request_text_char_count,response_text_preview,response_text_sha256,response_text_char_count,total_input_tokens,total_output_tokens,total_cost,error_message
a370560f-bcbe-40dc-bfbd-07f1b7253186,2026-04-15T01:35:29.299767Z,ci-smoke-q-183adc03-135e-4f5c-b30a-d88d603ff16b,ci-smoke-session-f9c4c65d-6313-4836-b7f2-7e9233b65a84,resp_0e7291af8d15de9d0069deeb2ff66481979a22511c28ce9b01,M0042150,success,48,0,72,5203,b0c3c4c8144b407e8c7c24d8cd983054,/Shared/Benefit_Quote_MVP1,databricks,gpt-5.2,prod,ci-smoke,CI Smoke,What is my copayment or cost for seeing my primary care physician (PCP)?,dae292e8730498cbed2112d7c3444ec4af6d9a7baa2c860ddc63b0435e652b08,72,"**Brief Answer** For an in-network primary care physician (PCP) visit, the cost is a **$25 copay per visit**. Out-of-network PCP visits are **50% coinsurance** and are **subject to the out-of-network deductible**. --- **Detailed Explanation** ### A. Primary care physician (PCP) office visit (Generalist) #### **In-Network Provider** For in-network providers: the plan has a **$350** deductible per person, not to exceed **$700** per family. [5] However, primary care physician (PCP) office visit is not",7feaaca1099e350baf122d36813ed9794c8980fbff8de0f168fa915fc2e21a85,5203,111595,2583,0.231453,null
aab0d2da-cca8-4d03-a4c4-3779473d0569,2026-04-15T01:04:15.886776Z,ci-smoke-q-3a309144-bf32-451c-94d4-8eaefed6f108,ci-smoke-session-a5dabb08-e8d3-4569-9aaf-5d61bd1a022d,resp_0428c5e01611153c0069dee3c3a68c8194b0fb2da187826013,M0042150,success,75,0,72,5141,3f1916f5d3b54aa1ac57b5e944379dc9,/Shared/Benefit_Quote_MVP1,databricks,gpt-5.2,prod,ci-smoke,CI Smoke,What is my copayment or cost for seeing my primary care physician (PCP)?,dae292e8730498cbed2112d7c3444ec4af6d9a7baa2c860ddc63b0435e652b08,72,"**Brief Answer** For this PPO plan, a **PCP (generalist) office visit** with an **in-network provider** has a **$25 copay per visit**. If the member sees an **out-of-network provider**, the member pays **50% coinsurance after the out-of-network deductible** (and the provider may balance bill). --- **Detailed Explanation** ## 1) Primary care physician (PCP) / Generalist office visit *(Office Visit/Consultation – Physician Care, Generalist)* ### **In-Network Provider** For in-network providers: the pla",e623ef66f070c421fc0aed4e1a85c1be89bcc2773ed75166dcc8812e035ca168,5141,111595,4241,0.254665,null
53e40846-fb4f-4cf7-958b-aa5e5146ab60,2026-04-15T00:41:32.343547Z,27cdae0a-f958-4853-995f-97b0b11f9ef1,393fb955-54f3-4421-aa32-c8b48f307626,resp_0a682552c2e92ce10069dede232d8c8195b1be7e11d54404df,ML002185,success,152,0,26,20435,bf1888fc15d84366b8ad855f4c8e6d62,/Shared/Benefit_Quote_MVP1,databricks,gpt-5.2,prod,rgaban31,rgaban31@blueshieldca.com,prescription drug benefits,9ba9a32b8fe44eaa0314d37dce6890e51f77816fe78c327fdaa98281f08d8648,26,"**Brief Answer** For this PPO plan, outpatient prescription drugs are covered when filled at a **Participating Pharmacy**, with copays/coinsurance based on the drug tier. **Retail 90-day supplies are not covered** under this plan’s outpatient pharmacy benefit (Tier 1–4). --- **Detailed Explanation** ### Prescription Drug Benefits — Tier summary (Participating Pharmacy) Retail (30-day supply) - Tier 1: $8 per prescription - Tier 2: $25 per prescription - Tier 3: $45 per prescription - Tier",da3d0cbcb3541fe0e561456dae6e3e4694b482a56a9a8e4d7e2ca2f74cf517fd,20435,109331,10206,0.325544,null
4d179fad-04ce-491e-ba7e-2fbd30fbb71d,2026-04-15T00:35:15.802284Z,a1e88ac2-a3fa-4b46-9767-4f64644ba334,a08603e0-e44b-4b32-882f-45c67e8e2cc6,resp_0742dbcdcf3ef1c30069dedccd45888195abd78ae2723d3bd9,ML002175,success,117,0,7,5090,9d63457dcb4e4429ab22d0a3446b747a,/Shared/Benefit_Quote_MVP1,databricks,gpt-5.2,prod,rluang01,rluang01@blueshi